In [1]:
import torch

import cheetah

In [2]:
incoming = cheetah.ParticleBeam.from_parameters(
    num_particles=100_000,
    sigma_x=torch.tensor(1e-3),
    sigma_y=torch.tensor(1e-3),
    energy=torch.tensor(1e9),
)
histogram_screen = cheetah.Screen(
    resolution=(500, 500),
    pixel_size=torch.tensor((2e-5, 2e-5)),
    is_active=True,
    method="histogram",
)
kde_screen = cheetah.Screen(
    resolution=(500, 500),
    pixel_size=torch.tensor((2e-5, 2e-5)),
    is_active=True,
    method="kde",
)
cic_screen = cheetah.Screen(
    resolution=(500, 500),
    pixel_size=torch.tensor((2e-5, 2e-5)),
    is_active=True,
    method="cloud-in-cell",
)

In [3]:
%%timeit
histogram_screen.track(incoming)
reading = histogram_screen.reading

2.97 ms ± 125 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [4]:
%%timeit
kde_screen.track(incoming)
reading = kde_screen.reading

159 ms ± 2.75 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [5]:
%%timeit
cic_screen.track(incoming)
reading = cic_screen.reading

3.9 ms ± 366 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [6]:
vectorized_incoming = cheetah.ParticleBeam.from_parameters(
    num_particles=10_000,
    sigma_x=torch.linspace(1e-3, 2e-3, 100),
    sigma_y=torch.tensor(1e-3),
    energy=torch.tensor(1e9),
)

In [7]:
%%timeit
kde_screen.track(vectorized_incoming)
reading = kde_screen.reading

1.27 s ± 46.4 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [8]:
%%timeit
cic_screen.track(vectorized_incoming)
reading = cic_screen.reading

33.2 ms ± 1.24 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [9]:
tiny_incoming = cheetah.ParticleBeam.from_parameters(
    num_particles=10,
    sigma_x=torch.tensor(1e-3),
    sigma_y=torch.tensor(1e-3),
    energy=torch.tensor(1e9),
)
tiny_histogram_screen = cheetah.Screen(
    resolution=(3, 4),
    pixel_size=torch.tensor((2e-5, 2e-5)),
    is_active=True,
    method="histogram",
)
tiny_kde_screen = cheetah.Screen(
    resolution=(3, 4),
    pixel_size=torch.tensor((2e-5, 2e-5)),
    is_active=True,
    method="kde",
)
tiny_cic_screen = cheetah.Screen(
    resolution=(3, 4),
    pixel_size=torch.tensor((2e-5, 2e-5)),
    is_active=True,
    method="cloud-in-cell",
)

In [10]:
%%timeit
tiny_histogram_screen.track(tiny_incoming)
reading = tiny_histogram_screen.reading

215 μs ± 467 ns per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [11]:
%%timeit
tiny_kde_screen.track(tiny_incoming)
reading = tiny_kde_screen.reading

624 μs ± 41.9 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [12]:
%%timeit
tiny_cic_screen.track(tiny_incoming)
reading = tiny_cic_screen.reading

771 μs ± 38.9 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [13]:
cuda_incoming = incoming.to(device="cuda", dtype=torch.float32)
cuda_histogram_screen = histogram_screen.to(device="cuda", dtype=torch.float32)
cuda_kde_screen = kde_screen.to(device="cuda", dtype=torch.float32)
cuda_cic_screen = cic_screen.to(device="cuda", dtype=torch.float32)

In [14]:
# %%timeit
# cuda_histogram_screen.track(cuda_incoming)
# reading = cuda_histogram_screen.reading

In [15]:
%%timeit
cuda_kde_screen.track(cuda_incoming)
reading = cuda_kde_screen.reading

5.04 ms ± 5.97 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [16]:
%%timeit
cuda_cic_screen.track(cuda_incoming)
reading = cuda_cic_screen.reading

1.95 ms ± 1.52 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [17]:
cuda_vectorized_incoming = vectorized_incoming.to(device="cuda", dtype=torch.float32)

In [18]:
%%timeit
cuda_kde_screen.track(cuda_vectorized_incoming)
reading = cuda_kde_screen.reading

38.5 ms ± 128 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [19]:
%%timeit
cuda_cic_screen.track(cuda_vectorized_incoming)
reading = cuda_cic_screen.reading

2.77 ms ± 3.11 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [20]:
cuda_tiny_incoming = tiny_incoming.to(device="cuda", dtype=torch.float32)
cuda_tiny_histogram_screen = tiny_histogram_screen.to(device="cuda", dtype=torch.float32)
cuda_tiny_kde_screen = tiny_kde_screen.to(device="cuda", dtype=torch.float32)
cuda_tiny_cic_screen = tiny_cic_screen.to(device="cuda", dtype=torch.float32)

In [21]:
%%timeit
cuda_tiny_histogram_screen.track(cuda_tiny_incoming)
reading = cuda_tiny_histogram_screen.reading

NotImplementedError: Could not run 'aten::_histogramdd_from_bin_tensors' with arguments from the 'CUDA' backend. This could be because the operator doesn't exist for this backend, or was omitted during the selective/custom build process (if using custom build). If you are a Facebook employee using PyTorch on mobile, please visit https://fburl.com/ptmfixes for possible resolutions. 'aten::_histogramdd_from_bin_tensors' is only available for these backends: [CPU, Meta, BackendSelect, Python, FuncTorchDynamicLayerBackMode, Functionalize, Named, Conjugate, Negative, ZeroTensor, ADInplaceOrView, AutogradOther, AutogradCPU, AutogradCUDA, AutogradHIP, AutogradXLA, AutogradMPS, AutogradIPU, AutogradXPU, AutogradHPU, AutogradVE, AutogradLazy, AutogradMTIA, AutogradMAIA, AutogradPrivateUse1, AutogradPrivateUse2, AutogradPrivateUse3, AutogradMeta, AutogradNestedTensor, Tracer, AutocastCPU, AutocastMTIA, AutocastMAIA, AutocastXPU, AutocastMPS, AutocastCUDA, FuncTorchBatched, BatchedNestedTensor, F

In [ ]:
%%timeit
cuda_tiny_kde_screen.track(cuda_tiny_incoming)
reading = cuda_tiny_kde_screen.reading

In [ ]:
%%timeit
cuda_tiny_cic_screen.track(cuda_tiny_incoming)
reading = cuda_tiny_cic_screen.reading